<a href="https://colab.research.google.com/github/zhouning/alphaearth-training-system/blob/master/colab/train_paper58_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paper 5+8 × Paper 12 Ablation — Prithvi-on-AlphaEarth-areas

Drop-in replacement encoder ablation for Paper 5+8: extract Prithvi-100M features from HLS multi-spectral on the same 17 study areas Paper 5+8 used (15 from `world_model.py:DEFAULT_TRAINING_AREAS` + bishan + banzhucun), retrain `LatentDynamicsNet` at `z_dim=768`, compare against the AlphaEarth 64-dim encoder. Adds one ablation row to Paper 5+8 Table 1.

**Before running:**
1. Drive root: `paper58_ablation.zip` (built locally; 20 KB)
2. Runtime → Change runtime type → **L4 GPU**
3. Register an Earth Engine Cloud project at https://code.earthengine.google.com/register (one-time, free, takes 30 s). Note the project ID.
4. Edit cell 6 below — replace `EE_PROJECT = 'ee-yourname'` with your real project ID.
5. Run cells top-to-bottom. Phase A (cell 7) ≈ 4h, Phase C (cell 9) ≈ 1.5h on L4.

**Outputs (synced to Drive `paper58_ablation_results/`):**
- `data/prithvi/{area}_emb_{year}.npy` — Phase A Prithvi features, 17×8=136 files
- `weights/latent_dynamics_prithvi_768d_seed{42,123,456}.pt` — Phase C LDN checkpoints
- `results/encoder_comparison.json` — Phase B static cosine + LULC probe
- `results/paper8_ablation_encoder.json` — Phase D final metrics for Table 1

In [44]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/paper58_ablation'
DRIVE_RESULTS = '/content/drive/MyDrive/paper58_ablation_results'
os.makedirs(WORK, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
%cd $WORK

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/paper58_ablation


In [45]:
# 2. GPU check
!nvidia-smi | head -20

Fri May 15 05:53:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [46]:
# 3. Unzip ablation scripts (small, ~20 KB)
import shutil, subprocess
shutil.copy('/content/drive/MyDrive/paper58_ablation.zip', '/content/paper58_ablation.zip')
subprocess.run(['unzip', '-qo', '/content/paper58_ablation.zip', '-d', WORK], check=True)
!ls -la

total 84
drwxr-xr-x 4 root root  4096 May 15 05:53 .
drwxr-xr-x 1 root root  4096 May 15 04:40 ..
-rw-rw-rw- 1 root root  4323 May 15  2026 colab_extract_prithvi.py
-rw-rw-rw- 1 root root  6916 May 15  2026 compare_encoders.py
drwxr-xr-x 4 root root  4096 May 15 04:52 data
-rw-rw-rw- 1 root root  8674 May 15  2026 eval_prithvi_vs_alphaearth.py
-rw-rw-rw- 1 root root 13808 May 15  2026 extract_prithvi_embeddings.py
drwxr-xr-x 4 root root  4096 May 15 05:53 geoadapter
-rw-rw-rw- 1 root root  2888 May 15  2026 paper58_runtime.py
-rw-rw-rw- 1 root root  2492 May 15  2026 README_PRITHVI_ABLATION.md
-rw-rw-rw- 1 root root  8349 May 15  2026 train_prithvi_ldn.py


In [47]:
!sha256sum /content/drive/MyDrive/paper58_ablation.zip

d972b4e55b3d19e5e931966e71887af16562add533251a8f83fbea4169efb90f  /content/drive/MyDrive/paper58_ablation.zip


In [48]:
# 4. Install deps (Prithvi forward needs torch+rasterio; HLS needs ee)
!pip install -q earthengine-api rasterio scikit-learn

In [49]:
# 5. Download Prithvi weights from HuggingFace (350 MB, ~1 min on Colab).
# If Drive has a copy, use it first; else fetch from HF.
import os, shutil
WEIGHTS_DST = f'{WORK}/data/weights/prithvi/Prithvi_100M.pt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/Prithvi_100M.pt'
os.makedirs(os.path.dirname(WEIGHTS_DST), exist_ok=True)
if not os.path.exists(WEIGHTS_DST):
    if os.path.exists(DRIVE_WEIGHTS):
        shutil.copy(DRIVE_WEIGHTS, WEIGHTS_DST)
        print('Copied from Drive')
    else:
        !wget -q https://huggingface.co/ibm-nasa-geospatial/Prithvi-100M/resolve/main/Prithvi_100M.pt -O $WEIGHTS_DST
        print('Downloaded from HuggingFace')
!ls -la $WEIGHTS_DST

-rw-r--r-- 1 root root 453671052 May 15 04:46 /content/paper58_ablation/data/weights/prithvi/Prithvi_100M.pt


In [50]:
# 6. Configure environment + authenticate Earth Engine
#    EE_PROJECT defaults to the user's registered project. Override here if needed.
EE_PROJECT = 'ee-zn19860115'

os.environ['ALPHAEARTH_REPO'] = WORK
os.environ['PRITHVI_WEIGHTS'] = WEIGHTS_DST
os.environ['DATA_AGENT_DIR'] = '/dev/null'  # force fallback area list
os.environ['EE_PROJECT'] = EE_PROJECT       # so extract_prithvi_embeddings.py sees it too

import ee
try:
    ee.Initialize(project=EE_PROJECT)
    print(f'GEE initialized (project={EE_PROJECT})')
except Exception:
    ee.Authenticate()  # opens browser flow, paste auth code back into Colab
    ee.Initialize(project=EE_PROJECT)
    print(f'GEE authenticated (project={EE_PROJECT})')

GEE initialized (project=ee-zn19860115)


In [ ]:
# 7. PHASE A — extract Prithvi features for 17 areas × 8 years (~4 hours wall, ~80 Colab units)
# Restore any previously-extracted .npy files from Drive so reruns skip completed work.
import os, shutil
from pathlib import Path
src = Path(DRIVE_RESULTS) / 'data' / 'prithvi'
dst = Path(WORK) / 'data' / 'prithvi'
dst.mkdir(parents=True, exist_ok=True)
if src.exists():
    n = 0
    for f in src.glob('*.npy'):
        if not (dst / f.name).exists():
            shutil.copy(f, dst / f.name)
            n += 1
    print(f'restored {n} previous npy files')

!python extract_prithvi_embeddings.py

Extracting Prithvi features: 17 areas × 8 years
  yangtze_delta 2017: re-extracting (stub (1, 1, 768))
  yangtze_delta 2017:   HLS band convention: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7']
Prithvi loaded on cuda, weights=pretrained
shape=(24, 24, 768), 4.6s
  yangtze_delta 2018: re-extracting (stub (1, 1, 768))
  yangtze_delta 2018: shape=(24, 24, 768), 1.0s
  yangtze_delta 2019: re-extracting (stub (1, 1, 768))
  yangtze_delta 2019: shape=(24, 24, 768), 1.3s
  yangtze_delta 2020: re-extracting (stub (1, 1, 768))
  yangtze_delta 2020: shape=(24, 24, 768), 1.0s
  yangtze_delta 2021: re-extracting (stub (1, 1, 768))
  yangtze_delta 2021: shape=(24, 24, 768), 1.0s
  yangtze_delta 2022: re-extracting (stub (1, 1, 768))
  yangtze_delta 2022: shape=(24, 24, 768), 1.1s
  yangtze_delta 2023: re-extracting (stub (1, 1, 768))
  yangtze_delta 2023: shape=(24, 24, 768), 1.6s
  yangtze_delta 2024: re-extracting (stub (1, 1, 768))
  yangtze_delta 2024: shape=(24, 24, 768), 2.2s
  jing_jin_ji 2017: re-ex

In [ ]:
# 8. Sync Phase A output to Drive (in case Phase C crashes, you don't redo Phase A)
import shutil, os
src = f'{WORK}/data/prithvi'
dst = f'{DRIVE_RESULTS}/data/prithvi'
os.makedirs(dst, exist_ok=True)
for f in os.listdir(src):
    shutil.copy(os.path.join(src, f), os.path.join(dst, f))
!ls $dst | head -5
!ls $dst | wc -l

In [ ]:
# 9. PHASE C — retrain LatentDynamicsNet at z_dim=768 on Prithvi features
# 17 areas × 3 seeds × 50 epochs. Estimated ~1.5 h on L4.
!python train_prithvi_ldn.py --epochs 50 --seeds 42 123 456

In [ ]:
# 10. Sync Phase C checkpoints to Drive
import shutil, os
src = f'{WORK}/weights'
dst = f'{DRIVE_RESULTS}/weights'
os.makedirs(dst, exist_ok=True)
for f in os.listdir(src):
    shutil.copy(os.path.join(src, f), os.path.join(dst, f))
!ls $dst

In [ ]:
# 11. PHASE B — static encoder comparison (no LDN, fast). Skips if no AlphaEarth files on /data.
# Note: Phase B requires AlphaEarth .npy files in {WORK}/data/{area}_emb_{year}.npy. For a full
# Phase B run you must also stage those from D:/test/paper8/data/ to Drive and copy them in here.
# If you only ran Phase A+C, skip Phase B and run it locally instead (it's CPU-only).
!python compare_encoders.py || echo 'Phase B skipped (AlphaEarth files missing). Run locally.'

In [ ]:
# 12. PHASE D — final metric aggregation for Paper 5+8 Table 1
# Same caveat as Phase B: needs AlphaEarth side files. Run locally if absent.
!python eval_prithvi_vs_alphaearth.py || echo 'Phase D skipped (AlphaEarth files missing). Run locally.'

# Sync any results back to Drive
import shutil, os
src = f'{WORK}/results'
dst = f'{DRIVE_RESULTS}/results'
os.makedirs(dst, exist_ok=True)
if os.path.exists(src):
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
    !ls $dst

## Notes

- **Resume**: Cell 7 (`extract_prithvi_embeddings.py`) skips areas/years already on disk; Cell 9 (`train_prithvi_ldn.py`) writes per-seed checkpoints — restart at any seed by deleting `weights/latent_dynamics_prithvi_768d_seed{N}.pt`.
- **Drive sync**: Cells 8 and 10 sync intermediates so you don't lose work if a session is killed.
- **Phase B/D need AlphaEarth side**: They compare AE vs Prithvi pixel-wise. AE features live at `D:/test/paper8/data/{area}_emb_{year}.npy` locally — easier to run B+D on your local machine after pulling Phase A/C outputs from Drive.
- **Budget**: Phase A ~80 units, Phase C ~50 units (1.5h L4). Total ~130 units.
- **GEE project**: Earth Engine now requires a Cloud project. Free at https://code.earthengine.google.com/register — pick "Use without a Cloud project" for the simplest path.